# Chapter 05 — Transformer Block

The Transformer is the architecture behind every modern large language model — GPT-4, Claude, Llama, Gemini. In this chapter, you will assemble the complete Transformer decoder block by block, understanding not just what each component does, but why it's necessary. By the end, you'll build GPT-2 with 124 million parameters.

In [ ]:
import torch
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, mask=None):
        # Pre-LayerNorm + Residual Connection
        normed = self.ln1(x)
        attn_out, _ = self.attn(normed, normed, normed, attn_mask=mask)
        x = x + attn_out            # Residual

        normed = self.ln2(x)
        x = x + self.ffn(normed)     # Residual
        return x

# Demo: forward pass
block = TransformerBlock(d_model=64, n_heads=4, d_ff=256)
x = torch.randn(2, 10, 64)  # batch=2, seq=10, d_model=64
out = block(x)
params = sum(p.numel() for p in block.parameters())
print(f"Input shape:  {tuple(x.shape)}")
print(f"Output shape: {tuple(out.shape)}")
print(f"Parameters:   {params:,}")

---

**Course:** Aprenda LLM | **Chapter 05**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.